# Transformers & self-attention

_Modern Deep Learning & AI_

**Every token looks at every other token and pulls in what matters. No loops, all at once.**

A Transformer reads a whole sentence at once. It does not crawl word by word.

     The trick is self-attention. Every word builds three small vectors: a query (what am I looking for?), a key (what do I offer?), and a value (what I carry).

---

This notebook builds self-attention from scratch, one step at a time. Run each cell, read the note above it, then watch attention rediscover what "it" refers to in a real sentence. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Write the scaled dot-product attention function

Self-attention lets every token mix in information from every other token. Each token carries a **query** (what it is looking for), a **key** (what it offers), and a **value** (what it passes along). We score every query against every key with a dot product, divide by `sqrt(d)` to keep the numbers from blowing up, softmax each row into weights that sum to 1, and use those weights to blend the values.

In [ ]:
import torch
import torch.nn.functional as F

def attention(Q, K, V):
    # Q, K, V: (batch, seq_len, d). One query/key/value vector per token.
    d = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / d ** 0.5    # (batch, seq, seq)
    weights = F.softmax(scores, dim=-1)             # each row sums to 1
    blended = weights @ V                           # blended values
    return blended, weights

### Step 2 — Project the input into Q, K, V

In *self*-attention, the queries, keys, and values all come from the same input `x`. Three learnable weight matrices project that shared input into the three different roles. Here we use a tiny example: one batch of 4 tokens, each an 8-dimensional vector, with random projection matrices standing in for learned weights.

In [ ]:
torch.manual_seed(0)
batch, seq, d = 1, 4, 8                              # 4 tokens, dim 8
x = torch.randn(batch, seq, d)

# learnable projections build Q, K, V from the same input (self-attention)
Wq, Wk, Wv = (torch.randn(d, d) for _ in range(3))
Q = x @ Wq
K = x @ Wk
V = x @ Wv

### Step 3 — Run attention and check the shapes

Applying the function returns the blended output — same shape as the input, one updated vector per token — plus the attention weight matrix. Confirming that each row of weights sums to about 1 is a quick sanity check that the softmax did its job: every token spreads a total of one unit of attention across all tokens.

In [ ]:
out, attn = attention(Q, K, V)
print("output shape:", out.shape)                   # (1, 4, 8)
print("row 0 weights sum:", attn[0, 0].sum().item())  # ~1.0

## Visualize the data & results

_In the sentence The animal didnt cross the street because it was too tired, what does it attend to?_

### Step 1 — Embed a real sentence with a coreference

We take the sentence *"The animal didn't cross the street because it was too tired"* and give each token a random embedding. Then we deliberately bend two embeddings so that *"it"* points back at *"animal"* and *"the"* points back at *"The"*. This plants a coreference relationship we hope attention will recover.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# scaled dot-product attention over a REAL sentence
tokens = ["The", "animal", "didnt", "cross", "the", "street",
          "because", "it", "was", "too", "tired"]
seq, d = len(tokens), 16
rng = np.random.default_rng(7)

# one embedding per token
X = np.stack([np.random.default_rng(1000 + i).standard_normal(d)
              for i in range(seq)])
X[7] = 0.30 * X[7] + 0.95 * X[1]                    # coreference: "it" embedding ~ "animal"
X[4] = 0.30 * X[4] + 0.95 * X[0]                    # "the" ~ "The" 

### Step 2 — Compute the attention weight matrix

We build queries and keys with near-identity projections (so the planted similarities survive), score every query against every key, scale by `sqrt(d)`, and softmax each row. Subtracting the row max before exponentiating is the standard numerical-stability trick — it leaves the softmax result unchanged but prevents overflow.

In [ ]:
Wq = np.eye(d) + 0.15 * rng.standard_normal((d, d))   # near-identity projections
Wk = np.eye(d) + 0.15 * rng.standard_normal((d, d))
Q = X @ Wq
K = X @ Wk

scores = Q @ K.T / np.sqrt(d)
scores -= scores.max(axis=1, keepdims=True)          # numerical-stability shift
W = np.exp(scores)
W /= W.sum(axis=1, keepdims=True)                     # each row sums to 1

### Step 3 — Plot the attention heatmap

Each row of the heatmap shows where one token sends its attention. Look at the row for *"it"*: because we tied its embedding to *"animal"*, the brightest cell in that row should land on the *"animal"* column — attention has discovered the coreference.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(W, cmap="viridis", vmin=0, vmax=1)
ax.set_xticks(range(seq))
ax.set_xticklabels(tokens, rotation=90)
ax.set_yticks(range(seq))
ax.set_yticklabels(tokens)
ax.set_title("Self-attention: row it peaks on animal")
fig.colorbar(im, ax=ax)
plt.show()